# LegalQA Task 2 — Canonical Reproducible Dual-T4 Kaggle Pipeline
End-to-end LegalQA training, validation, and inference pipeline on Kaggle Dual NVIDIA T4 GPUs.
- **Lifecycle**: Configuration -> Environment & Secrets -> Packaged Code/Data Resolution -> Preflight -> Reranker Fine-Tuning -> QLoRA Fine-Tuning & Reload Smoke Test -> Dev Evaluation -> Dual-GPU Inference -> Strict 1000-ID Submission Validation.
- **Hardware**: Dual NVIDIA T4 (GPU 0: Qwen Generator | GPU 1: Dense Retriever + BGE Reranker | CPU: BM25S + QA Memory + Selector).

In [ ]:
# Cell 1 — Configuration Flags & Settings
SEED = 42
RUN_RERANKER_TRAINING = False       # Set True to fine-tune BGE Reranker on Kaggle
RUN_GENERATOR_TRAINING = False      # Set True to fine-tune Qwen2.5 with QLoRA on Kaggle
RUN_DEV_EVALUATION = True           # Run quick regression evaluation on validation fold
RUN_PUBLIC_INFERENCE = True         # Run public test inference and generate submission.json
REUSE_EXISTING_CHECKPOINTS = False  # Reuse pre-trained checkpoints if available
ALLOW_INDEX_REBUILD = False         # Allow building dense embeddings on the fly if missing
FINAL_STACK = "stack_a"             # 'stack_a' (DEk21 + Qwen3B) or 'stack_b' (BGE-M3 + Qwen1.5B)

print("=== Pipeline Execution Mode Configuration ===")
print(f"Reranker Training:     {RUN_RERANKER_TRAINING}")
print(f"Generator Training:    {RUN_GENERATOR_TRAINING}")
print(f"Dev Evaluation:        {RUN_DEV_EVALUATION}")
print(f"Public Inference:      {RUN_PUBLIC_INFERENCE}")
print(f"Selected Final Stack:  {FINAL_STACK}")

In [ ]:
# Cell 2 — Environment & Safe Secrets
import os, sys, gc, glob, json, zipfile, re, math, time, subprocess
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# Set deterministic seeds
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Retrieve HuggingFace Token safely from Kaggle Secrets (never printed or logged)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    if HF_TOKEN:
        os.environ["HF_TOKEN"] = HF_TOKEN
        os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
        print("HF_TOKEN securely retrieved from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("HF_TOKEN found in environment variables.")
    else:
        print("Notice: HF_TOKEN secret not found; using public weights.")

# Dual-GPU Device Allocation
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
if gpu_count >= 2:
    GEN_DEVICE = "cuda:0"
    RETRIEVAL_DEVICE = "cuda:1"
elif gpu_count == 1:
    GEN_DEVICE = "cuda:0"
    RETRIEVAL_DEVICE = "cuda:0"
else:
    GEN_DEVICE = "cpu"
    RETRIEVAL_DEVICE = "cpu"

print(f"CUDA GPUs Detected: {gpu_count}")
for i in range(gpu_count):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} | VRAM: {p.total_memory / (1024**3):.1f} GB | Compute: sm_{p.major}{p.minor}")
print(f"Hardware Plan -> Generator: {GEN_DEVICE} | Dense & Reranker: {RETRIEVAL_DEVICE}")

In [ ]:
# Cell 3 — Resolve Packaged Source, Data & Models
# 1. Resolve Code Root
code_candidates = glob.glob("/kaggle/input/**/code/LegalQA", recursive=True) + [".", "/kaggle/working", "/kaggle/working/LegalQA"]
resolved_code_root = None
for cand in code_candidates:
    if os.path.exists(os.path.join(cand, "src")) and os.path.exists(os.path.join(cand, "configs")):
        resolved_code_root = os.path.abspath(cand)
        if resolved_code_root not in sys.path:
            sys.path.insert(0, resolved_code_root)
        print(f"Packaged Code Root resolved: {resolved_code_root}")
        break

# 2. Resolve Data Artifacts
qa_files = glob.glob("/kaggle/input/**/qa_unique.parquet", recursive=True) or glob.glob("artifacts/**/qa_unique.parquet", recursive=True)
chunks_files = glob.glob("/kaggle/input/**/legal_chunks.parquet", recursive=True) or glob.glob("artifacts/**/legal_chunks.parquet", recursive=True)
known_files = glob.glob("/kaggle/input/**/known_qa.json", recursive=True) or glob.glob("artifacts/**/known_qa.json", recursive=True)
test_files = glob.glob("/kaggle/input/**/public-official.json", recursive=True) or glob.glob("artifacts/**/public-official.json", recursive=True)

assert qa_files, "qa_unique.parquet not found!"
assert chunks_files, "legal_chunks.parquet not found!"
assert known_files, "known_qa.json not found!"
assert test_files, "public-official.json not found!"

QA_PATH = qa_files[0]
CHUNKS_PATH = chunks_files[0]
KNOWN_QA_PATH = known_files[0]
TEST_PATH = test_files[0]
DATA_DIR = os.path.dirname(QA_PATH)

# 3. Resolve Index Directories
bm25_dirs = [d for d in glob.glob("/kaggle/input/**/bm25*", recursive=True) if os.path.isdir(d)] or ["artifacts/task2/indexes/bm25"]
dek21_dirs = [d for d in glob.glob("/kaggle/input/**/dek21*", recursive=True) if os.path.isdir(d)] or ["artifacts/task2/indexes/dek21"]
BM25_DIR = bm25_dirs[0] if bm25_dirs and os.path.exists(bm25_dirs[0]) else "artifacts/task2/indexes/bm25"
DEK21_DIR = dek21_dirs[0] if dek21_dirs and os.path.exists(dek21_dirs[0]) else "artifacts/task2/indexes/dek21"

# 4. Resolve Mounted Qwen Model
qwen_configs = glob.glob("/kaggle/input/**/qwen*3b*/**/config.json", recursive=True) + glob.glob("/kaggle/input/**/3b-instruct*/**/config.json", recursive=True)
if qwen_configs:
    MODEL_PATH = os.path.dirname(qwen_configs[0])
else:
    MODEL_PATH = "Qwen/Qwen2.5-3B-Instruct"

print(f"Data Directory: {DATA_DIR}")
print(f"BM25 Index:     {BM25_DIR}")
print(f"Dense Index:    {DEK21_DIR}")
print(f"Qwen Base Path: {MODEL_PATH}")

In [ ]:
# Cell 4 — Preflight Validation
from scripts.preflight_kaggle import run_preflight_checks
from scripts.audit_parameters import audit_parameter_budget

preflight_res = run_preflight_checks(
    pipeline_config_path="configs/pipeline.yaml" if os.path.exists("configs/pipeline.yaml") else os.path.join(resolved_code_root, "configs/pipeline.yaml"),
    models_config_path="configs/models.yaml" if os.path.exists("configs/models.yaml") else os.path.join(resolved_code_root, "configs/models.yaml"),
    require_cuda=torch.cuda.is_available(),
    expected_gpu_count=2 if torch.cuda.device_count() >= 2 else 1,
    check_dataset_files=False,
    public_path=TEST_PATH,
    stack=FINAL_STACK,
)

if not preflight_res["passed"]:
    print("Preflight errors detected:", preflight_res["errors"])
    if not ALLOW_INDEX_REBUILD:
        raise RuntimeError(f"PREFLIGHT FAILED: {preflight_res['errors']}")
else:
    print("Preflight validation completely PASSED!")

In [ ]:
# Cell 5 — Load Data & Index Metadata
from src.task2.qa_memory import QAMemory
from src.common.bm25 import BM25Retriever

print("Loading verified QA Memory...")
memory = QAMemory.load(KNOWN_QA_PATH, QA_PATH)
print(f"Loaded QA Memory: {len(memory.id_to_answer):,} IDs | {len(memory.question_to_answer):,} unique questions.")

print("Loading BM25 Index metadata...")
if os.path.exists(os.path.join(BM25_DIR, "bm25_manifest.json")) or os.path.exists(os.path.join(BM25_DIR, "bm25s_index")):
    bm25 = BM25Retriever.load(BM25_DIR, corpus_path=CHUNKS_PATH)
else:
    print("Building BM25 index on the fly...")
    df_chunks = pd.read_parquet(CHUNKS_PATH)
    bm25 = BM25Retriever()
    bm25.fit(df_chunks.to_dict("records"))
print(f"BM25 Ready: {bm25.corpus_size:,} chunks indexed.")

In [ ]:
# Cell 6 — Optional Reranker Fine-Tuning
RERANKER_CHECKPOINT = "BAAI/bge-reranker-v2-m3"

if RUN_RERANKER_TRAINING:
    from src.task2.training.train_reranker import train_bge_reranker
    pairs_path = os.path.join(DATA_DIR, "reranker_training_pairs.parquet")
    reranker_out = "/kaggle/working/checkpoints/reranker/best"
    print(f"Starting Reranker fine-tuning on {RETRIEVAL_DEVICE}...")
    res_rerank = train_bge_reranker(
        pairs_path=pairs_path,
        output_dir=reranker_out,
        model_name="BAAI/bge-reranker-v2-m3",
        epochs=2,
        device=RETRIEVAL_DEVICE,
    )
    if res_rerank["status"] == "completed":
        RERANKER_CHECKPOINT = reranker_out
        print(f"Using trained reranker checkpoint: {RERANKER_CHECKPOINT}")
else:
    print(f"Reranker training skipped. Using checkpoint: {RERANKER_CHECKPOINT}")

In [ ]:
# Cell 7 — Optional QLoRA Training & Reload Smoke Test
ADAPTER_PATH = None

if RUN_GENERATOR_TRAINING:
    from src.task2.training.train_generator import run_qlora_training
    labels_path = os.path.join(DATA_DIR, "retrieval_labels.parquet")
    qlora_out = "/kaggle/working/checkpoints/generator/hf_adapter"
    print(f"Starting QLoRA fine-tuning on {GEN_DEVICE}...")
    res_qlora = run_qlora_training(
        model_name=MODEL_PATH,
        qa_path=QA_PATH,
        labels_path=labels_path,
        chunks_path=CHUNKS_PATH,
        output_dir=qlora_out,
        epochs=1,
        batch_size=1,
        grad_accum=8,
        lr=2e-4,
        device=GEN_DEVICE,
    )
    if res_qlora["status"] == "completed":
        ADAPTER_PATH = qlora_out
        print(f"Using trained QLoRA adapter: {ADAPTER_PATH}")
else:
    print("QLoRA training skipped. Using base generator.")

In [ ]:
# Cell 8 — Development Evaluation
if RUN_DEV_EVALUATION:
    from scripts.run_oof_validation import run_oof_validation
    print("Running development evaluation sample (50 items across folds)...")
    eval_res = run_oof_validation(
        qa_path=QA_PATH,
        chunks_path=CHUNKS_PATH,
        bm25_dir=BM25_DIR,
        dek21_dir=DEK21_DIR,
        eval_output_dir="/kaggle/working/evaluations",
        num_eval_samples=50,
        mode="fast",
        model_path=MODEL_PATH,
        adapter_path=ADAPTER_PATH,
        gen_device=GEN_DEVICE,
        retrieval_device=RETRIEVAL_DEVICE,
    )
    print(f"Dev Sample METEOR: {eval_res['mean_meteor']:.4f}")
else:
    print("Dev evaluation skipped.")

In [ ]:
# Cell 9 — Load Final Inference Pipeline on Dual-T4
from src.task2.predict import LegalQAPipeline
from src.common.dense import DenseRetriever
from src.common.reranker import BGEReranker
from src.task2.evidence_packer import EvidencePacker
from src.task2.generator import QwenGenerator
from src.task2.selector import CandidateSelector

print("Initializing final Dual-T4 pipeline components...")
# 1. Dense Retriever on GPU 1
if os.path.exists(os.path.join(DEK21_DIR, "embeddings.npy")):
    print(f"Loading FP16 Dense Index on {RETRIEVAL_DEVICE}...")
    dense = DenseRetriever.load_index(DEK21_DIR, corpus_path=CHUNKS_PATH, device=RETRIEVAL_DEVICE)
else:
    print(f"FINAL_PIPELINE_ERROR: Dense embeddings not found at {DEK21_DIR}.", file=sys.stderr)
    if not ALLOW_INDEX_REBUILD:
        raise RuntimeError(f"Missing dense index at {DEK21_DIR}.")
    dense = DenseRetriever(device=RETRIEVAL_DEVICE)
    dense.fit(bm25.corpus)

# 2. Reranker on GPU 1
print(f"Loading Cross-Encoder Reranker ({RERANKER_CHECKPOINT}) on {RETRIEVAL_DEVICE}...")
reranker = BGEReranker(model_name=RERANKER_CHECKPOINT, device=RETRIEVAL_DEVICE)

# 3. Evidence Packer on CPU
packer = EvidencePacker(bm25.corpus)

# 4. Qwen Generator on GPU 0
print(f"Loading Qwen Generator on {GEN_DEVICE}...")
generator = QwenGenerator.load(
    model_path=MODEL_PATH,
    adapter_path=ADAPTER_PATH,
    device=GEN_DEVICE,
    runtime="torch" if torch.cuda.is_available() else "fallback",
    fail_on_fallback=True if torch.cuda.is_available() else False,
)

# 5. Selector on CPU
selector = CandidateSelector(policy="fixed_baseline", best_fixed_candidate="stitched_extract")

pipeline = LegalQAPipeline(memory, bm25, dense, reranker, packer, generator, selector)
print("Final Dual-T4 Pipeline loaded successfully!")

In [ ]:
# Cell 10 — Load Public Test Set & Memory Pre-pass
with open(TEST_PATH, "r", encoding="utf-8") as f:
    public_test = json.load(f)

print(f"Loaded {len(public_test)} public test questions from {TEST_PATH}.")
submission = {}
unseen_items = []

# Exact QA Memory Pre-pass
for qid, item in public_test.items():
    q_text = str(item.get("question", "")).strip()
    exact_ans = memory.lookup_exact(qid, q_text)
    if exact_ans:
        submission[str(qid)] = {"answer": exact_ans}
    else:
        unseen_items.append((str(qid), q_text))

print(f"Exact Memory Hits: {len(submission):,} | Questions to Retrieve & Generate: {len(unseen_items):,}")

In [ ]:
# Cell 11 & 12 — Batched Retrieval, Evidence Packing, Generation & Selection
start_time = time.time()
candidate_selection_counts = defaultdict(int)

if unseen_items and RUN_PUBLIC_INFERENCE:
    batch_size = 4 if torch.cuda.is_available() else 1
    print(f"Executing inference pipeline on {len(unseen_items)} queries (Batch Size = {batch_size})...")

    for i in tqdm(range(0, len(unseen_items), batch_size), desc="Dual-T4 Inference"):
        batch_chunk = unseen_items[i:i + batch_size]
        for qid, q in batch_chunk:
            selected, cands, ev = pipeline.predict_single(qid, q, max_new_tokens=384, return_candidates=True)
            submission[qid] = {"answer": selected}
            # Track candidate selection
            matched_name = "custom"
            for c_name, c_val in cands.items():
                if c_val.strip() == selected.strip():
                    matched_name = c_name
                    break
            candidate_selection_counts[matched_name] += 1

elapsed = time.time() - start_time
print(f"Public inference completed in {elapsed:.1f}s.")

In [ ]:
# Cell 13 — Strict Submission Verification & Diagnostics
print("=== Strict Submission Verification ===")
assert len(submission) == 1000, f"Submission item count mismatch! Expected 1000, got {len(submission)}"

test_keys = set(public_test.keys())
sub_keys = set(submission.keys())
assert test_keys == sub_keys, f"Submission keys do not match public test keys! Diff: {test_keys ^ sub_keys}"

for qid, val in submission.items():
    ans = val.get("answer", "")
    assert isinstance(ans, str) and len(ans.strip()) > 0, f"Empty answer for ID {qid}!"
    assert "[DOCUMENT]" not in ans and "[ARTICLE]" not in ans, f"Internal tag found in ID {qid}!"

lengths = [len(v["answer"].split()) for v in submission.values()]
print(f"Total Queries:        {len(submission):,}")
print(f"Mean Word Count:      {np.mean(lengths):.1f} words")
print(f"Median Word Count:    {np.median(lengths):.1f} words")
print(f"P90 Word Count:       {np.percentile(lengths, 90):.1f} words")
print(f"Min / Max Length:     {np.min(lengths)} / {np.max(lengths)} words")

print("\nCandidate Selection Breakdown:")
for k, v in sorted(candidate_selection_counts.items(), key=lambda x: -x[1]):
    print(f" - {k:22s}: {v:,} ({v/max(1, len(unseen_items))*100:.1f}%)")

In [ ]:
# Cell 14 — Save Artifacts & Generate Submission Zip
out_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "artifacts/task2/submissions"
os.makedirs(out_dir, exist_ok=True)

out_json = os.path.join(out_dir, "submission.json")
out_zip = os.path.join(out_dir, "submission.json.zip")
run_manifest_path = os.path.join(out_dir, "run_manifest.json")

with open(out_json, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(out_json, arcname="submission.json")

run_manifest = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "num_queries": len(submission),
    "final_stack": FINAL_STACK,
    "reranker_checkpoint": RERANKER_CHECKPOINT,
    "adapter_path": ADAPTER_PATH,
    "candidate_counts": dict(candidate_selection_counts),
    "mean_word_count": float(np.mean(lengths)),
}
with open(run_manifest_path, "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, indent=2)

print(f"Saved final submission.json ({os.path.getsize(out_json)/1024:.1f} KB)")
print(f"Saved final submission.json.zip ({os.path.getsize(out_zip)/1024:.1f} KB)")
print(f"Saved run_manifest.json ({run_manifest_path})")
print("\nSUCCESS: All LegalQA pipeline stages completed and verified!")